In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, silhouette_score
from sklearn.decomposition import PCA
from scipy.stats import chi2
from sklearn.mixture import GaussianMixture

In [2]:
def matriz_conf(y_true, y_pred, labels):
    total_labels = labels
    print(total_labels)
    cm = np.zeros((len(total_labels),len(total_labels)), dtype=int)
    for i in range(len(y_true)):
        cm[y_true[i]][y_pred[i]] += 1

    
    cm = pd.DataFrame(cm, columns=total_labels, index=total_labels)

    cm_transp = pd.DataFrame(np.transpose(cm.to_numpy()), columns=total_labels, index=total_labels)

    for c in cm_transp.columns:
        cm_transp[c] = cm_transp[c]/cm_transp[c].sum()

    cm_porcento = pd.DataFrame(np.transpose(cm_transp.to_numpy()), columns=total_labels, index=total_labels)

    return cm, cm_porcento

def acc(cm, hidden_classes):
    cm_transp = pd.DataFrame(np.transpose(cm.dropna().to_numpy()), columns=cm.columns, index=cm.columns)
    acc = 0
    total = 0
    for c in cm_transp.columns:
        if c not in hidden_classes:
            acc += cm_transp[c][c]
        else:
            acc += cm_transp[c][-1]
        total += cm_transp[c].sum()
    return acc/total

In [3]:
filenames = [0,2,3,4,5]

labels_str = ['DDoS', 'Benign', 'DoS', 'BruteForce', 'Bot', 'Web']

filenames

# pd.set_option('future.no_silent_downcasting', True)

[0, 2, 3, 4, 5]

In [5]:
exp_val = []
y_true_all_exp_val = []
for i in range(len(filenames)):
    exp_val.append(pd.read_csv(f'val_{filenames[i]}_GMM_BIC_1_10_scores_corr.csv'))
    y_true_all_exp_val.append(exp_val[i]['Label'].values.tolist())
    exp_val[i] = exp_val[i].drop(columns=['Label'])

In [6]:
from sklearn.metrics import classification_report
ths = [19, 20, 21, 22, 23, 24, 25, 26]
f1s = [[],[],[],[],[],[],[],[]]
for i in range(len(exp_val)):
    index = 0
    for th in ths:
        y_pred = []
        prog= 0
        for j in range(len(exp_val[i])):
            # print(exp[i][j])
            m = np.nanmax(exp_val[i].values[j])
            # print(m)
            if m > th:
                y_pred.append(exp_val[i].values[j].tolist().index(m))
            else:
                y_pred.append(-1)
        # print(y_pred[:10])

        cm, cm_porcento = matriz_conf(y_true_all_exp_val[i], y_pred, list(range(len(labels_str))) + [-1])
        print(f'th = {th} hidden = {filenames[i]}')
        display(cm)
        tp = cm[-1][filenames[i]]
        fp = cm[-1].sum() - tp
        fn = cm.iloc[filenames[i]].sum() - tp
        tn = cm.drop(columns=[-1]).values.sum() - fn

        acc = (tp+tn)/(tp+fp+tn+fn)
        recall = tp/(tp+fn)
        precision = tp/(tp+fp)
        if precision == 0 or recall == 0:
            f1 = 0
        else:
            f1 = 2*precision*recall/(precision+recall)

        f1s[index].append(f1)
        index += 1
        print(f'th: {th} hidden: {filenames[i]} acc:{acc} recall:{recall} precision:{precision} f1:{f1}')

        true_labels = np.array(y_true_all_exp_val[i])
        true_labels[true_labels == filenames[i]] = -1

        print('MULTICLASS DETECTION')
        print(classification_report(true_labels, y_pred))

[0, 1, 2, 3, 4, 5, -1]
th = 19 hidden = 0


,0,1,2,3,4,5,-1
0,0,14538,0,0,0,0,187690
1,0,124675,14,2,172,1262,2411
2,0,1,82106,0,0,0,198
3,0,0,3,60914,0,0,33
4,0,51,0,0,45562,0,177
5,0,12,0,0,9,98,28
-1,0,0,0,0,0,0,0


th: 19 hidden: 0 acc:0.9665644785327989 recall:0.928110845184643 precision:0.9850580202270425 f1:0.9557368910162565
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.99      0.93      0.96    202228
           1       0.90      0.97      0.93    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      1.00      1.00     45790
           5       0.07      0.67      0.13       147

    accuracy                           0.96    519956
   macro avg       0.82      0.93      0.84    519956
weighted avg       0.97      0.96      0.96    519956

[0, 1, 2, 3, 4, 5, -1]
th = 20 hidden = 0


,0,1,2,3,4,5,-1
0,0,9604,0,0,0,0,192624
1,0,123091,14,2,162,1260,4007
2,0,1,81938,0,0,0,366
3,0,0,3,60914,0,0,33
4,0,45,0,0,45550,0,195
5,0,8,0,0,8,77,54
-1,0,0,0,0,0,0,0


th: 20 hidden: 0 acc:0.9725765257060213 recall:0.9525090491920011 precision:0.9764039760947694 f1:0.9643085102388694
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.98      0.95      0.96    202228
           1       0.93      0.96      0.94    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.99      1.00     45790
           5       0.06      0.52      0.10       147

    accuracy                           0.97    519956
   macro avg       0.83      0.90      0.83    519956
weighted avg       0.97      0.97      0.97    519956

[0, 1, 2, 3, 4, 5, -1]
th = 21 hidden = 0


...

th: 24 hidden: 5 acc:0.9826254529229397 recall:0.10204081632653061 precision:0.00168218010541662 f1:0.003309796999117387
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.10      0.00       147
           0       1.00      1.00      1.00    202228
           1       1.00      0.96      0.98    128536
           2       1.00      1.00      1.00     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.94      0.97     45790

    accuracy                           0.98    519956
   macro avg       0.83      0.83      0.82    519956
weighted avg       1.00      0.98      0.99    519956

[0, 1, 2, 3, 4, 5, -1]
th = 25 hidden = 5


,0,1,2,3,4,5,-1
0,201302,0,0,0,0,0,926
1,97,121198,37,1,10,0,7193
2,0,0,81022,0,0,0,1283
3,0,0,3,60922,0,0,25
4,1,12,0,0,41012,0,4765
5,0,96,0,0,7,0,44
-1,0,0,0,0,0,0,0


th: 25 hidden: 5 acc:0.9725072890783066 recall:0.29931972789115646 precision:0.0030907558302894073 f1:0.006118334144476118
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.30      0.01       147
           0       1.00      1.00      1.00    202228
           1       1.00      0.94      0.97    128536
           2       1.00      0.98      0.99     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.90      0.94     45790

    accuracy                           0.97    519956
   macro avg       0.83      0.85      0.82    519956
weighted avg       1.00      0.97      0.99    519956

[0, 1, 2, 3, 4, 5, -1]
th = 26 hidden = 5


,0,1,2,3,4,5,-1
0,201190,0,0,0,0,0,1038
1,93,117876,8,1,10,0,10548
2,0,0,79555,0,0,0,2750
3,0,0,3,60920,0,0,27
4,0,12,0,0,40956,0,4822
5,0,95,0,0,7,0,45
-1,0,0,0,0,0,0,0


th: 26 hidden: 5 acc:0.9629064767018748 recall:0.30612244897959184 precision:0.00234009360374415 f1:0.004644681839294008
MULTICLASS DETECTION
              precision    recall  f1-score   support

          -1       0.00      0.31      0.00       147
           0       1.00      0.99      1.00    202228
           1       1.00      0.92      0.96    128536
           2       1.00      0.97      0.98     82305
           3       1.00      1.00      1.00     60950
           4       1.00      0.89      0.94     45790

    accuracy                           0.96    519956
   macro avg       0.83      0.85      0.81    519956
weighted avg       1.00      0.96      0.98    519956



# Média absolute threshold

In [7]:
for i in range(len(f1s)):
    print(f'Média f1 absolute th {ths[i]}: {np.mean(np.array(f1s[i]))}')

Média f1 absolute th 19: 0.5759053499165432
Média f1 absolute th 20: 0.5854567987381418
Média f1 absolute th 21: 0.6712250410475109
Média f1 absolute th 22: 0.6615088692267281
Média f1 absolute th 23: 0.6393720651054047
Média f1 absolute th 24: 0.6482698638556869
Média f1 absolute th 25: 0.6114025899499536
Média f1 absolute th 26: 0.5522381589841714
